In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pickle
from tqdm import tqdm
import random
import os
from scipy.stats import norm
from decimal import Decimal, getcontext
from base_model import float_to_binary_array_not_IEEE, create_dense_model
import matplotlib.pyplot as plt

#load the data
data=np.load('jordan_data.npz',allow_pickle=True)
signals = data['signals']
times = data['times']

# each index is a different time series... pick an index from 0-99
index=2

X = times[index].copy()
# Normalize
X = X/X.max()
# Use new base 2 embedding
X = np.array([float_to_binary_array_not_IEEE(X[i]) for i in range(len(X))])
Y = signals[index]
validation_split = int(0.8*X.shape[0])

# separate into train and test
Xtrain=X[:validation_split]
Xval=X[validation_split:]
Ytrain=Y[:validation_split]
Yval=Y[validation_split:]

strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

# Wrap the model creation and compilation in the strategy's scope
with strategy.scope():
    model = create_dense_model(dropout_frac=0.5,n_streams=20)
    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=.01), loss='mae')

# save the weights at the lowest val_loss
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint("embed_time_predict_dense.weights.h5", save_best_only=True, save_weights_only=True, monitor='val_loss')

# train
history = model.fit(
    Xtrain,
    Ytrain,
    epochs=10000,
    validation_data=(Xval,Yval),
    callbacks=[checkpoint_callback],
    batch_size=512,
    shuffle=True
)

# plot

model.load_weights('embed_time_predict_dense.weights.h5')
p0=model.predict(Xtrain,batch_size=256)
p1=model.predict(Xval,batch_size=256)

plt.figure(figsize=(15,5))
plt.plot(times[index],Y,label='truth')
plt.plot(times[index][:validation_split],p0[:,0],label='train prediction')
plt.plot(times[index][validation_split:],p1[:,0],label='val prediction')


plt.savefig('timepred37.png')
plt.close()
